Deflection of a plate using the FSDT
---
Engineering Mechanics Stability

Graduate School Course

Author: Saullo G. P. Castro

Date: 8 July 2025



In [1]:
import numpy as np
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import eigsh
from composites import isotropic_plate

from legendre import vecf, vecfxi, vecfxixi
from legendre_gauss_quadrature import legendre_gauss_quadrature

# approximation order
m1 = m2 = 20
N = 3*m1*m2
print('m1 = %d' % m1)
print('m2 = %d' % m2)
print('N = %d' % N)
i = np.arange(m1)
j = np.arange(m2)
pts1, weights1 = legendre_gauss_quadrature(2*m1 - 1)
pts2, weights2 = legendre_gauss_quadrature(2*m2 - 1)

# Material properties
E = 200.e9
nu = 0.3
G = E/(2*(1 + nu))

# Boundary conditions w (simply supported)
wxit1 = 0
wxir1 = 0
wxit2 = 0
wxir2 = 0
wetat1 = 0
wetar1 = 0
wetat2 = 0
wetar2 = 0

# Boundary conditions phix phiy
xit1 = 1
xir1 = 1
xit2 = 1
xir2 = 1
etat1 = 1
etar1 = 1
etat2 = 1
etar2 = 1

# Geometric properties
a = 3
b = 7
h = 0.005

# Calculating ABD matrices
prop = isotropic_plate(thickness=h, E=E, nu=nu)

# Applied load
Pforce = 1.


Swx = np.zeros(N)
Swy = np.zeros(N)
Sphix = np.zeros(N)
Sphiy = np.zeros(N)
Sphixx = np.zeros(N)
Sphixy = np.zeros(N)
Sphiyx = np.zeros(N)
Sphiyy = np.zeros(N)

    
buff = np.zeros((N, N))
K = np.zeros((N, N))
Fext = np.zeros((N,))

def addouter(matrix, vec1, vec2):
    np.outer(vec1, vec2, out=buff)
    matrix += buff

# stiffness matrix
# numerical integration in 2D using Legendre-Gauss quadrature
for xi, wxi in zip(pts1, weights1):
    wP_xi = vecf(m1, xi, wxit1, wxir1, wxit2, wxir2)
    wPx_xi = vecfxi(m1, xi, wxit1, wxir1, wxit2, wxir2)
    
    P_xi = vecf(m1, xi, xit1, xir1, xit2, xir2)
    Px_xi = vecfxi(m1, xi, xit1, xir1, xit2, xir2)
    
    for eta, weta in zip(pts2, weights2):
        wP_eta = vecf(m2, eta, wetat1, wetar1, wetat2, wetar2)
        wPx_eta = vecfxi(m2, eta, wetat1, wetar1, wetat2, wetar2)

        P_eta = vecf(m2, eta, etat1, etar1, etat2, etar2)
        Px_eta = vecfxi(m2, eta, etat1, etar1, etat2, etar2)

        weight = wxi*weta

        Pi, Pj = np.meshgrid(P_xi, P_eta, indexing='ij')  
        Sphix[m1*m2:2*m1*m2] = (Pi*Pj).flatten()
        Sphiy[2*m1*m2:] = (Pi*Pj).flatten()
        
        Pxi, Pj = np.meshgrid(wPx_xi, wP_eta, indexing='ij')
        Swx[:m1*m2] = (Pxi*Pj*(2/a)).flatten()
        
        Pxi, Pj = np.meshgrid(Px_xi, P_eta, indexing='ij')
        Sphixx[m1*m2:2*m1*m2] = (Pxi*Pj*(2/a)).flatten()
        Sphiyx[2*m1*m2:] = (Pxi*Pj*(2/a)).flatten()
        
        Pi, Pxj = np.meshgrid(wP_xi, wPx_eta, indexing='ij')
        Swy[:m1*m2] = (Pi*Pxj*(2/b)).flatten()
        
        Pi, Pxj = np.meshgrid(P_xi, Px_eta, indexing='ij')
        Sphixy[m1*m2:2*m1*m2] = (Pi*Pxj*(2/b)).flatten()
        Sphiyy[2*m1*m2:] = (Pi*Pxj*(2/b)).flatten()
        
        e1xx = Sphixx
        e1yy = Sphiyy
        e1xy = Sphixy + Sphiyx
        
        g0yz = Sphiy + Swy
        g0xz = Sphix + Swx
        
        Mxx = prop.D11*e1xx + prop.D12*e1yy + prop.D16*e1xy
        Myy = prop.D12*e1xx + prop.D22*e1yy + prop.D26*e1xy
        Mxy = prop.D16*e1xx + prop.D26*e1yy + prop.D66*e1xy

        Qy = prop.A44*g0yz + prop.A45*g0xz
        Qx = prop.A45*g0yz + prop.A55*g0xz
        
        detJ = a*b/4
        
        # stiffness matrix
        addouter(K, detJ*weight*Mxx, e1xx)
        addouter(K, detJ*weight*Myy, e1yy)
        addouter(K, detJ*weight*Mxy, e1xy)
        addouter(K, detJ*weight*Qy, g0yz)
        addouter(K, detJ*weight*Qx, g0xz)
        

# external force vector
xi = 0 # x = a/2
eta = 0 # y = b/2
P_xi = vecf(m1, xi, xit1, xir1, xit2, xir2)
P_eta = vecf(m2, eta, etat1, etar1, etat2, etar2)
Pi, Pj = np.meshgrid(P_xi, P_eta, indexing='ij')  
Sw = np.zeros(N)
Sw[:m1*m2] = (Pi*Pj).flatten()
Fext = Pforce*Sw


m1 = 20
m2 = 20
N = 1200


In [2]:
from scipy.sparse.linalg import spsolve
from scipy.sparse import csc_matrix

#eigvals, eigvecs = eigsh(A=csc_matrix(KG), k=3, which='SM',
#                         M=csc_matrix(KNL), tol=0, sigma=1., mode='cayley')
from structsolve import solve
u = solve(csc_matrix(K), Fext)

			Removing null columns...
				144 columns removed
			finished!


In [3]:
wcentre = Sw@u
print('w at centre', wcentre)
print('w reference', 6.594931610258557e-05) # from https://github.com/saullocastro/pyfe3d/blob/main/tests/test_quad4_static_point_load.py

w at centre 2.778138036216032e-05
w reference 6.594931610258557e-05
